# Apache Iceberg 中文入门

本 Notebook 对应当前项目的 `Spark + Iceberg REST Catalog + MinIO` 环境。按顺序运行每个单元格，即可理解建表、写入、模式演进、快照和时间旅行。

- **Spark**：执行 SQL 和计算。
- **Iceberg**：定义表格式、快照和并发提交规则。
- **REST Catalog**：保存表名、命名空间与当前元数据位置。
- **MinIO**：保存 Parquet 数据文件和 Iceberg 元数据文件。

## 1. 启动 Spark 并检查 Catalog

普通 Python 内核不会自动创建 Spark 会话，因此先显式启动它。项目的 Iceberg、REST Catalog 和 MinIO 参数会从 `spark-defaults.conf` 自动加载；Catalog 名称为 `lake`。

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName('Iceberg中文入门')
         .getOrCreate())
print(f'Spark {spark.version} 已启动')

Spark 3.5.5 已启动


26/09/17 14:53:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SHOW NAMESPACES IN lake").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|lake         |
|spark_catalog|
+-------------+

+---------+
|namespace|
+---------+
|learning |
|yigraph  |
+---------+



## 2. 创建第一张 Iceberg 表

`days(created_at)` 是隐藏分区。查询者只需要过滤时间列，不需要手工拼接分区目录。

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lake.learning")
spark.sql("""
CREATE OR REPLACE TABLE lake.learning.orders (
  order_id BIGINT,
  customer_id STRING,
  amount DECIMAL(12, 2),
  created_at TIMESTAMP
) USING iceberg
PARTITIONED BY (days(created_at))
""")

## 3. 写入和查询

一次成功写入会生成新的快照。数据文件不会被原地修改。

In [ ]:
spark.sql("""
INSERT INTO lake.learning.orders VALUES
  (1, 'c-101', 128.50, TIMESTAMP '2026-09-17 09:00:00'),
  (2, 'c-102',  66.00, TIMESTAMP '2026-09-17 10:15:00'),
  (3, 'c-101', 249.90, TIMESTAMP '2026-09-18 08:30:00')
""")
spark.sql("SELECT * FROM lake.learning.orders ORDER BY order_id").show(truncate=False)

## 4. 模式演进

Iceberg 通过列 ID 跟踪字段，增加、重命名或重排字段时不需要重写全部历史数据。

In [ ]:
spark.sql("ALTER TABLE lake.learning.orders ADD COLUMN channel STRING")
spark.sql("UPDATE lake.learning.orders SET channel = 'web' WHERE channel IS NULL")
spark.sql("SELECT * FROM lake.learning.orders ORDER BY order_id").show(truncate=False)

## 5. 查看快照与数据文件

Iceberg 的元数据表可以像普通表一样查询。`snapshots` 展示版本历史，`files` 展示当前快照引用的数据文件。

In [ ]:
spark.sql("""
SELECT snapshot_id, committed_at, operation
FROM lake.learning.orders.snapshots
ORDER BY committed_at
""").show(truncate=False)

spark.sql("""
SELECT file_path, record_count, file_size_in_bytes
FROM lake.learning.orders.files
""").show(truncate=False)

## 6. 时间旅行

下面读取最早的快照。它发生在增加 `channel` 字段之前，因此该字段在旧数据上显示为空。

In [ ]:
snapshots = spark.sql("""
SELECT snapshot_id
FROM lake.learning.orders.snapshots
ORDER BY committed_at
""").collect()

first_snapshot_id = snapshots[0]['snapshot_id']
print(f'读取快照：{first_snapshot_id}')
(spark.read
      .option('snapshot-id', str(first_snapshot_id))
      .table('lake.learning.orders')
      .orderBy('order_id')
      .show(truncate=False))

## 小结

你已经完成了 Iceberg 最核心的工作流：建表、写入、查询、模式演进、观察快照和读取历史版本。下一步请打开 `02-YiGraph图数据建模.ipynb`。